# Week 60, Day 2: Decorators I

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w060d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def loud(func):
    def wrapper(name):
        return func(name).upper() + "!"
    return wrapper

def greet(name):
    return f"hello, {name}"

print(greet("ada"))
greet = loud(greet)
print(greet("ada"))

@loud
def farewell(name):
    return f"goodbye, {name}"

print(farewell("ben"), farewell.__name__)

In [ ]:
import time

def timed(func):
    print("decorating", func.__name__)       # runs once, at the def
    def wrapper(n):
        start = time.perf_counter()
        result = func(n)
        print(f"{func.__name__}({n}) took under a second: {time.perf_counter() - start < 1}")
        return result
    return wrapper

@timed
def total(n):
    return sum(range(n))

print("defined")
print(total(1000), total(10))

In [ ]:
def logged(func):
    def wrapper(*args, **kwargs):
        shown = [repr(a) for a in args] + [f"{k}={v!r}" for k, v in kwargs.items()]
        result = func(*args, **kwargs)
        print(f"{func.__name__}({', '.join(shown)}) -> {result!r}")
        return result
    return wrapper

@logged
def price(amount, vat=0.2, *, rounding=2):
    return round(amount * (1 + vat), rounding)

price(10)
price(9.99, vat=0.05)
price(3, rounding=0)

In [ ]:
import functools

def counted(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return func(*args, **kwargs)
    wrapper.calls = 0
    return wrapper

@counted
def square(x):
    """Return x squared."""
    return x * x

square(3)
square(4)
print(square.calls, square.__name__, square.__doc__, square.__wrapped__(5))

In [ ]:
def bold(func):
    def wrapper():
        return "<b>" + func() + "</b>"
    return wrapper

def italic(func):
    def wrapper():
        return "<i>" + func() + "</i>"
    return wrapper

@bold
@italic
def hello():
    return "hello"

print(hello())

commands = {}

def command(func):
    commands[func.__name__] = func
    return func                       # not wrapped: just recorded

@command
def stock():
    return "4 teas"

@command
def help_text():
    return "commands: " + ", ".join(sorted(commands))

print(commands["stock"](), commands["help_text"](), stock())

> **Key idea.** A decorator takes a function and returns one to use in its place; @deco above a def means f = deco(f), run once when the def runs. A wrapping decorator returns a closure that accepts *args and **kwargs, calls the original with them, and returns its result. Put functools.wraps(func) on every wrapper so the name, docstring and __wrapped__ survive; keep counters and other state as attributes of the wrapper. Stacked decorators apply bottom up and their wrappers run top down. A registering decorator records the function and returns it unchanged.

### Exercise: A first decorator

Write `shout(func)`: a decorator whose wrapper calls the function with any arguments and returns its result upper-cased with `"!"` added. Use `functools.wraps`. Then decorate `greet(name, greeting="hello")` (returning `f"{greeting}, {name}"`) with it using `@`.

In [ ]:
import functools

def shout(func):
    ___

def greet(name, greeting="hello"):
    return f"{greeting}, {name}"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_shout():\n    \"a wrapping decorator\"\n    assert greet(\"ada\") == \"HELLO, ADA!\" and greet(\"bo\", greeting=\"hi\") == \"HI, BO!\"\n    assert greet.__name__ == \"greet\" and greet.__wrapped__(\"x\") == \"hello, x\"\n    @shout\n    def nothing():\n        return \"ok\"\n    assert nothing() == \"OK!\" and \"@shout\" in source")

### Exercise: Remembering every call

Write `recorded(func)`: a decorator that keeps, as `wrapper.history`, a list of `(args, kwargs, result)` for every call (args as a tuple, kwargs as a dict), and `wrapper.calls`, the number of calls. If the function raises, record nothing for that call and let the exception pass through.

In [ ]:
import functools

def recorded(func):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_history():\n    \"state on the wrapper\"\n    @recorded\n    def add(a, b=0):\n        return a + b\n    assert add(1, 2) == 3 and add(5, b=1) == 6 and add.calls == 2\n    assert add.history == [((1, 2), {}, 3), ((5,), {\"b\": 1}, 6)] and add.__name__ == \"add\"\n    @recorded\n    def fail(x):\n        raise ValueError(x)\n    try:\n        fail(1)\n        assert False\n    except ValueError:\n        pass\n    assert fail.history == [] and fail.calls == 1")

### Exercise: Checking arguments

Write `positive(func)`: a decorator that raises `ValueError(f"{func.__name__}: argument {i} must be positive")` if positional argument i (from 0) is a number that isn't greater than 0, before calling the function. Then `typed(func)`: a decorator that checks each positional argument against the function's **annotations** (`func.__annotations__`, Week 33), in parameter order (use `func.__code__.co_varnames`), raising `TypeError(f"{name} must be {type.__name__}")` for a mismatch. Parameters without an annotation aren't checked.

In [ ]:
import functools

def positive(func):
    ___

def typed(func):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_checked():\n    \"validating decorators\"\n    @positive\n    def area(w, h):\n        return w * h\n    assert area(2, 3) == 6\n    try:\n        area(2, -1)\n        assert False\n    except ValueError as e:\n        assert str(e) == \"area: argument 1 must be positive\"\n    @typed\n    def label(name: str, count: int, note=None):\n        return f\"{name} x{count}\"\n    assert label(\"tea\", 3) == \"tea x3\" and label(\"jam\", 1, note=[1]) == \"jam x1\" and label(\"a\", 2, 99) == \"a x2\"\n    try:\n        label(\"tea\", \"3\")\n        assert False\n    except TypeError as e:\n        assert str(e) == \"count must be int\"\n    assert label.__name__ == \"label\"")

### Exercise: A registry of handlers

Write `handlers = {}` and a decorator `on(func)` that registers each function under its name **with any leading `"on_"` removed** (so `on_save` is registered as `"save"`) and returns it unchanged. Then `fire(event, *args)`: call the handler for that event with the arguments and return its result, or raise `KeyError(event)`.

In [ ]:
handlers = {}

def on(func):
    ___

def fire(event, *args):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_register():\n    \"registering\"\n    @on\n    def on_save(name):\n        return f\"saved {name}\"\n    @on\n    def quit():\n        return \"bye\"\n    assert handlers[\"save\"] is on_save and fire(\"save\", \"notes.txt\") == \"saved notes.txt\" and fire(\"quit\") == \"bye\"\n    assert on_save(\"x\") == \"saved x\", \"registering returns the function unchanged\"\n    try:\n        fire(\"open\")\n        assert False\n    except KeyError:\n        pass")

### Exercise: Which runs first?

Write two decorators: `tag(func)` wraps the result in `"[" + result + "]"`, and `trim(func)` strips spaces from the result. Then decorate `messy()` and `messy2()` (both returning `"  hi  "`) with the same two decorators in **opposite** orders, so that `messy() == "[hi]"` (trimmed, then tagged) and `messy2() == "[  hi  ]"` (tagged first, so trimming finds no spaces at the ends).

In [ ]:
import functools

def tag(func):
    ___

def trim(func):
    ___

def messy():
    return "  hi  "

def messy2():
    return "  hi  "

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_order():\n    \"stacking order\"\n    assert messy() == \"[hi]\" and messy2() == \"[  hi  ]\"\n    assert messy.__name__ == \"messy\" and messy2.__name__ == \"messy2\"")

### Bug hunt: A wrapper that loses the answer

The wrapper calls the function but forgets to **return** its result, so every decorated function returns `None` and the last line prints `None`. Return the result so it prints `25`.

In [ ]:
def announced(func):
    def wrapper(*args):
        print("calling", func.__name__)
        func(*args)
    return wrapper

@announced
def square(x):
    return x * x

print(square(5))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"return the result\"\n    expect_output(\"calling square\\n25\")")

### Bug hunt: Every function called wrapper

Without `functools.wraps`, the decorated function's name and docstring are the wrapper's, so this prints `wrapper None`. Add `@functools.wraps(func)` to the wrapper so it prints `area Width times height.`

In [ ]:
import functools

def counted(func):
    def wrapper(*args):
        wrapper.calls += 1
        return func(*args)
    wrapper.calls = 0
    return wrapper

@counted
def area(w, h):
    """Width times height."""
    return w * h

print(area.__name__, area.__doc__)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"wraps\"\n    expect_output(\"area Width times height.\")")

### Bug hunt: Called at the wrong time

This decorator calls the function **while decorating** and returns its result, so `hello` becomes the string `"hi"` and calling it fails with `TypeError`. Return a wrapper instead, so it prints `before` then `hi`.

In [ ]:
def noisy(func):
    print("before")
    return func()

try:
    @noisy
    def hello():
        return "hi"
    print(hello())
except TypeError:
    print("TypeError")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a wrapper\"\n    expect_output(\"before\\nhi\")")

### Puzzle: A general wrapping decorator

Arrange the lines of a decorator that times any function and returns its result. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # @a above @b above def f means f = ...: write it as a string like "a(b(f))"
q = ___   # True or False: a decorator runs each time the decorated function is called
r = ___   # the __name__ of a function decorated without functools.wraps, if the wrapper is called wrapper
s = ___   # what a registering decorator returns: "the function" or "a wrapper"
t = ___   # the attribute functools.wraps adds that holds the original function, as a string

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"a(b(f))\" and q is False and r == \"wrapper\", \"p, q and r\"\n    assert s == \"the function\" and t == \"__wrapped__\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a command router

A small text interface maps commands to functions. Write a class-free command system with decorators:

- `commands = {}`; a registering decorator `command(func)` storing the function under its name, and returning it unchanged.
- `dispatch(line)`: split the line into words; the first is the command name, the rest are its arguments as strings. Call the function and return its result. An unknown command returns `"unknown command: NAME"`; calling with the wrong number of arguments returns `"usage: NAME ARG1 ARG2 ..."` with the function's parameter names (use `func.__code__.co_varnames` and `co_argcount`; parameters with defaults are optional, shown in square brackets).
- A wrapping decorator `numbers(func)` that converts every argument to `float` before the call, returning `"error: arguments must be numbers"` if one can't be converted. It must keep the original's name and parameters for `dispatch` to read (use `functools.wraps`, and read the parameters from `func.__wrapped__` when it's present).
- `help_text()`: registered as a command named `help_text`, returning the sorted command names joined by `", "`, each followed by its docstring's first line in brackets if it has one.

Register these commands: `add` (two numbers, their sum), `greet(name, greeting="hello")` and `help_text`.

In [ ]:
import functools

commands = {}

def command(func):
    ___

def numbers(func):
    ___

def dispatch(line):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_router():\n    \"decorators together\"\n    assert dispatch(\"add 2 3.5\") == 5.5 and dispatch(\"greet Ada\") == \"hello, Ada\" and dispatch(\"greet Ben hi\") == \"hi, Ben\"\n    assert dispatch(\"add 2 x\") == \"error: arguments must be numbers\"\n    assert dispatch(\"add 2\") == \"usage: add a b\" and dispatch(\"greet\") == \"usage: greet name [greeting]\"\n    assert dispatch(\"fly away\") == \"unknown command: fly\"\n    assert commands[\"add\"].__name__ == \"add\"\n    h = dispatch(\"help_text\")\n    assert h.startswith(\"add (Add two numbers.)\") and \"greet\" in h and \"help_text\" in h")

## Warm-up

`@loud` above `def farewell` does exactly what `greet = loud(greet)` did. And the decorated function's name is now `wrapper`: today's fourth section fixes that.

## A function in, a function out

Functions are values (Week 7): they can be passed to functions and returned from them. A **decorator** is a function that takes a function and returns a (usually new) function to use in its place. The returned function is normally a **closure** (yesterday) that remembers the original and calls it, doing something before, after or around the call:

```python
def timed(func):
    def wrapper(n):
        start = time.perf_counter()
        result = func(n)                 # call the original
        print(f"{func.__name__} took {time.perf_counter() - start:.4f}s")
        return result                    # and hand back its result
    return wrapper
```

Three things a wrapper must remember to do: call the original (unless deliberately replacing it), pass the arguments on, and **return** its result. Forgetting the `return` is the classic decorator bug: the decorated function then returns `None`.

## The @ line

```python
@timed
def total(n):
    ...
```

means exactly:

```python
def total(n):
    ...
total = timed(total)
```

The decorator runs **once**, when the `def` statement runs (at import time for a module), not each time the function is called. Each **call** then runs the wrapper. The `@` syntax (PEP 318, Python 2.4) only saves writing the name three times, but it puts the decoration where readers see it first, at the top.

## Wrapping any function

A wrapper that takes exactly one argument only fits functions with one argument. To wrap **any** function, accept anything and pass it all on with `*args` and `**kwargs` (Week 7): `def wrapper(*args, **kwargs): return func(*args, **kwargs)`. The wrapper can look at the arguments (to log or check them), change them, or change the result.

## Keeping the name

The wrapper replaces the function, so the decorated function's `__name__`, `__doc__` and other details are now the **wrapper's**: `help()` shows the wrong thing, error messages and logs name `wrapper`, and test tools get confused. `functools.wraps(func)`, put above the wrapper (it's itself a decorator, one that takes an argument: Thursday explains how), copies the original's name, docstring, module and more onto the wrapper, and stores the original as `wrapper.__wrapped__`. **Use it in every wrapping decorator.**

A wrapper is also a fine place to keep state, as an attribute: `wrapper.calls = 0` and `wrapper.calls += 1` inside (no `nonlocal` needed: it's an attribute of an object, not a variable).

## Stacking and registering

**Stacking**: decorators written one above another apply **bottom up** (the nearest to the `def` first), so

```python
@a
@b
def f(): ...
```

means `f = a(b(f))`. When the function is called, the **outermost** wrapper (from `a`) runs first, and calls the one from `b`, which calls `f`. Order matters: timing outside caching measures cache hits; caching outside timing doesn't time them at all.

**Registering**: a decorator doesn't have to wrap. It can record the function somewhere and return it **unchanged**: a dict of commands, a list of tests to run, a table of event handlers. `atexit.register` and many web frameworks' routing decorators work like this.

## Practice

## Quiz

## Daily challenge

## Before you go

Find three decorators you've used in earlier weeks (`@property`, `@dataclass`, `@cache`, `@total_ordering`, `@classmethod`) and, for each, say whether it wraps the function or does something else to it. Tomorrow: decorators that take arguments, decorators on classes, and how `@property` works.